# Домашнее задание: Physics-Informed Neural Networks (PINNs)
В этой домашней работе мы:
- Решим **прямую задачу**: получим решение уравнения теплопроводности
- Решим **обратную задачу**: восстановим функцию источника тепла из решения прямой задачи и из точного решения.

---

## Задача: Уравнение теплопроводности

Рассмотрим 1D уравнение теплопроводности:

$$
\frac{\partial u}{\partial t} = \frac{\partial^2 u}{\partial x^2} + \cos(2 \pi x), \quad x \in [0, 1], \ t \ge 0
$$

Начальные условия: $u(x, 0) = \sin^2(3 \pi x)$  
Граничные условия: $\left. \frac{\partial u(x, t)}{\partial x} \right|_{x=0} = \left. \frac{\partial u(x, t)}{\partial x} \right|_{x=1} = 0$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.autograd import grad

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Класс модели PINN

In [ ]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            #your_code_here
        )

    def forward(self, x):
        return self.model(x)

Выполнение уравнения

In [ ]:
def pde_residual(model, x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)
    inputs = torch.cat([x, t], dim=1)
    u = model(inputs)
    
    #your_code_here

    u_t = grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    return #your_code_here

def analytical_solution(x, t):
    return 1/2 * ( -torch.exp(- 4 * np.pi**2 * t) *torch.cos(2 * np.pi * x) / (2 * np.pi**2) + (2 * (np.pi)**2 + torch.cos(2 * np.pi * x)) / (2 * np.pi**2) - torch.exp(-36 * np.pi**2 * t) * torch.cos(6 * np.pi * x))

## **Прямая задача (Forward Problem)**

В прямой задаче решается классическая задача Коши для одномерного уравнения теплопроводности. Источник тепла известен.

Хотим найти функцию $u(x, t)$, удовлетворяющую начальным, граничным условиям и уравнению теплопроводности.

#### Метод:

* Нейросеть $u_\theta(x, t)$ обучается за счёт минимизации следующей функции потерь:

$$
\mathcal{L}_{\text{Total}} = \mathcal{L}_{\text{IC}} + \mathcal{L}_{\text{BC}} + \mathcal{L}_{\text{PDE}}
$$

где

$$
\mathcal{L}_{\text{IC}} = \frac{1}{N_{\text{IC}}} \sum_{i=1}^{N_{\text{IC}}} \left(u_\theta(x_i, 0) - \sin^2(3 \pi x)\right)^2
$$

$$
\mathcal{L}_{\text{BC}} = \frac{1}{N_{\text{BC}}} \sum_{i=1}^{N_{\text{BC}}} \left(\left. \left( \frac{\partial u_\theta(x_i, t_i)}{\partial x} \right|_{x=0}\right)^2 + \left. \left(\frac{\partial u_\theta(x_i, t_i)}{\partial x} \right|_{x=1} \right)^2\right)
$$

$$
\mathcal{L}_{\text{PDE}} = \frac{1}{N_{\text{PDE}}} \sum_{i=1}^{N_{\text{PDE}}} \left( \frac{\partial u_\theta}{\partial t}(x_i, t_i) - \frac{\partial^2 u_\theta}{\partial x^2}(x_i, t_i) - \cos(2 \pi x)\right)^2
$$



### Обучение Forward PINN (2 балла)

Обучите Forward PINN и восстановите $u(x,t)$, сравните его с точным решением. Обратите внимание, что в этом пункте для обучения PINN запрещено использовать аналитическое решение. Модель не должна знать ничего кроме уравнений и условий к нему.

In [ ]:
def initial_condition(x):
    return #your_code_here

def boundary_condition(t):
    return #your_code_here

def train_forward(model, n_ic=100, n_bc=100, n_pde=5000, epochs=3000, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    #your_code_here

    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()

        # IC loss

        loss_ic = #your_code_here

        # BC loss

        loss_bc = #your_code_here

        # PDE residual loss

        loss_pde = #your_code_here

        loss = loss_ic + loss_bc + loss_pde
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(f"[Epoch {epoch}] Loss: {loss.item():.5f}")

        losses.append(loss.item())

    return losses

In [ ]:
model_forward = PINN().to(device)

losses_forward = train_forward(model_forward)

plt.plot(losses_forward)
plt.title("Loss по эпохам (forward задача)", fontsize=14)
plt.xlabel("Epoch", fontsize=14)
plt.ylabel("Loss", fontsize=14)
plt.yscale("log")
plt.show()

In [ ]:
# Создание сетки
x_grid = torch.linspace(0, 1, 100).reshape(-1, 1)
t_grid = torch.linspace(0, 1, 100).reshape(-1, 1)
x_mesh, t_mesh = torch.meshgrid(x_grid.squeeze(), t_grid.squeeze(), indexing='ij')
x_flat = x_mesh.reshape(-1, 1).to(device)
t_flat = t_mesh.reshape(-1, 1).to(device)

# Аналитическое решение
u_exact = analytical_solution(x_flat, t_flat).cpu().detach().numpy().reshape(100, 100)

# PINN решение
with torch.no_grad():
    u_pred = model_forward(torch.cat([x_flat, t_flat], dim=1)).cpu().numpy().reshape(100, 100)

# Визуализация
fig, axs = plt.subplots(1, 3, figsize=(14, 5))

c1 = axs[0].imshow(u_exact, extent=[0,1,0,1], origin='lower', aspect='auto', cmap='viridis')
axs[0].set_title("Аналитическое решение u(x,t)")
fig.colorbar(c1, ax=axs[0])

c2 = axs[1].imshow(u_pred, extent=[0,1,0,1], origin='lower', aspect='auto', cmap='viridis')
axs[1].set_title("PINN решение u(x,t)")
fig.colorbar(c2, ax=axs[1])

c3 = axs[2].imshow(u_pred - u_exact, extent=[0,1,0,1], origin='lower', aspect='auto', cmap='inferno')
axs[2].set_title("Разница между аналитическим решением и PINN")
fig.colorbar(c3, ax=axs[2])

plt.tight_layout()
plt.show()

## **Обратная задача (Inverse Problem)**

В обратной задаче предполагается, что функция источника тепла неизвестна, и ее необходимо восстановить.

У нас есть наблюдаемые значения функции $u(x, t)$ в ограниченном числе точек: $$\{(x_i, t_i, u_i^{\text{obs}})\}_{i=1}^{N_{\text{obs}}}$$

Хотим восстановить $f(x)$ по наблюдаемым данным.

#### Метод:

Обучаем нейросеть $u_\theta(x, t)$ и оптимизируем параметр $f(x)$, минимизируя:

$$
\mathcal{L}_{\text{Total}} = \mathcal{L}_{\text{data}} + \mathcal{L}_{\text{PDE}}
$$

где

$$
\mathcal{L}_{\text{data}} = \frac{1}{N_{\text{obs}}} \sum_{i=1}^{N_{\text{obs}}} \left(u_\theta(x_i, t_i) - u_i^{\text{obs}}\right)^2
$$

$$
\mathcal{L}_{\text{PDE}} = \frac{1}{N_{\text{PDE}}} \sum_{i=1}^{N_{\text{PDE}}} \left( \frac{\partial u_\theta}{\partial t}(x_i, t_i) - \frac{\partial^2 u_\theta}{\partial x^2}(x_i, t_i) - f(x)\right)^2
$$



### Обучение Inverse PINN на основе аналитического результата (4 балла)

Обучите нейронную сеть восстанавливать $f(x)$ из аналитического решения. Можете пользоваться любыми хаками и улучшениями, но главное -- не показывать нейронной сети информацию об $f(x)$. 
P.S. возможно, вам потребуется поменять архитектуру нейронной сети, чтобы на выходе помимо $u(x,t)$ выдавалась $f(x)$. Можете пойти и другим путем, например, создать отдельную маленькую нейронную сеть для $f(x)$, которая будет принимать на вход, соответственно, только $x$. Feel free to experiment

In [ ]:
def generate_data_points(n_obs, n_pde):
    x_obs = torch.rand(n_obs, 1)
    t_obs = torch.rand(n_obs, 1)
    x_pde = torch.rand(n_pde, 1)
    t_pde = torch.rand(n_pde, 1)
    return x_obs, t_obs, x_pde, t_pde

In [ ]:
def train_inverse(model, n_obs=200, n_pde=5000, epochs=3000, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    x_obs, t_obs, x_pde, t_pde = generate_data_points(n_obs, n_pde)
    x_obs = x_obs.to(device)
    t_obs = t_obs.to(device)
    x_pde = x_pde.to(device)
    t_pde = t_pde.to(device)

    u_obs = analytical_solution(x_obs, t_obs).detach().to(device)

    #your_code_here

    losses= []

    for epoch in range(epochs):
        optimizer.zero_grad()

        res_pde = pde_residual(model, x_pde, t_pde)
        loss_pde = #your_code_here

        u_pred = model(torch.cat([x_obs, t_obs], dim=1))
        loss_data = #your_code_here

        loss = loss_pde + loss_data
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(f"[Epoch {epoch}] Loss: {loss.item():.5f}, alpha: {model.alpha.item():.5f}")

        losses.append(loss.item())

    return #your_code_here

#### Запуск и визуализация результатов

Визуализируйте результаты, нарисуйте на одном графике $f(x)$, полученную при помощи PINN и сравните его с точной функцией источника тепла $f(x) = \cos(2 \pi x)$.

In [ ]:
#your_code_here

### Inverse PINN на основе ответов Forward PINN (4 балла)

Обучите обратный PINN на основе ответов Forward PINN. Соответственно, Forward PINN знает об источнике тепла, а обратный PINN пытается его восстановить.

In [ ]:
#your_code_here